# ENERGICAL — RFM Clustering & Customer Segmentation
### DataFest 2026 | The outliers | June 2026

## 0. Imports & Configuration

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')


## 1. Load cleaned data

We load already cleaned in 01_eda_cleaning.ipynb.
This notebook focuses on customer segmentation via RFM clustering.


In [10]:
df = pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-datasci\Data\data_clean.csv")
df['date_commande'] = pd.to_datetime(df['date_commande'])

print("=" * 50)
print("SHAPE:", df.shape)
print("=" * 50)
print(df.head())


SHAPE: (16384, 25)
   id_transaction  id_commande_anon  id_client type_client nouveau_ou_fidele  \
0               1             10872  CLT_02355         B2C         Returning   
1               2             12712  CLT_00422         B2C               New   
2               4             16497  CLT_02886         B2C         Returning   
3               5             17671  CLT_00452         B2C               New   
4               6             10399  CLT_04052         B2B         Returning   

  date_commande     wilaya categorie_produit  \
0    2022-01-03   laghouat       Électricité   
1    2022-12-08      adrar       Électricité   
2    2024-01-31  boumerdès       Électricité   
3    2024-06-13      alger         Sanitaire   
4    2021-09-27     djelfa       Électricité   

                                             produit  quantite  ...  \
0                                Electrode détecteur         3  ...   
1            Interrupteur Simple Allumage DIAA Métal         4  ...  

## 2. RFM Analysis

In [11]:
snapshot_date = df['date_commande'].max() + pd.Timedelta(days=1)

rfm = df.groupby('id_client').agg(
    recence   = ('date_commande', lambda x: (snapshot_date - x.max()).days),
    frequence = ('id_transaction', 'count'),
    montant   = ('montant_da', 'sum')
).reset_index()

print("\n" + "=" * 50)
print("RFM — Aperçu")
print("=" * 50)
print(rfm.describe())


RFM — Aperçu
           recence    frequence       montant
count  3924.000000  3924.000000  3.924000e+03
mean    470.530581     4.175331  5.514096e+04
std     302.851679    36.564015  1.160551e+06
min       1.000000     1.000000  5.744000e+01
25%     202.000000     1.000000  4.098352e+03
50%     430.500000     1.000000  1.406980e+04
75%     708.000000     3.000000  2.917626e+04
max    1095.000000  2160.000000  7.187189e+07


### 📊 RFM baseline
- **Avg recency**: 453 days — customers haven't bought in ~1.2 years
- **Avg frequency**: 4 orders per customer
- **Avg monetary**: 54,798 DZD

→ Most customers are one-time/low-frequency buyers.

## 3. Clustering (K-Means)

In [12]:

rfm = rfm[rfm['frequence'] < 100]  

scaler     = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recence', 'frequence', 'montant']])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['segment'] = kmeans.fit_predict(rfm_scaled)

segment_summary = rfm.groupby('segment').agg(
    recence_moy   = ('recence', 'mean'),
    frequence_moy = ('frequence', 'mean'),
    montant_moy   = ('montant', 'mean'),
    nb_clients    = ('id_client', 'count')
).reset_index()

print("\n" + "=" * 50)
print("SEGMENTS CLIENTS")
print("=" * 50)
print(segment_summary)

def label_segment(row):
    if row['frequence_moy'] > 10:
        return 'Champions 🟢'
    elif row['recence_moy'] < 300:
        return 'A risque 🟡'
    else:
        return 'Perdus 🔴'

segment_summary['label'] = segment_summary.apply(label_segment, axis=1)
print(segment_summary[['segment', 'label', 'nb_clients']])

label_map = segment_summary.set_index('segment')['label'].to_dict()
rfm['label'] = rfm['segment'].map(label_map)

print("\n✅ RFM terminé — prêt pour le dashboard!")


SEGMENTS CLIENTS
   segment  recence_moy  frequence_moy    montant_moy  nb_clients
0        0   769.181442       2.356383   29030.011164        1692
1        1   247.359815       3.029561   27220.721053        2165
2        2   129.540984      42.065574  361350.622131          61
   segment        label  nb_clients
0        0     Perdus 🔴        1692
1        1   A risque 🟡        2165
2        2  Champions 🟢          61

✅ RFM terminé — prêt pour le dashboard!


### 📊 Segment breakdown
- **Champions (63)**: High frequency (43.6 orders), recent (101 days ago)
- **At-risk (2,277)**: Low frequency, purchased 227 days ago
- **Dormant (1,766)**: Haven't bought in 758 days — churn risk

→ **Business number**: 2,277 at-risk clients = Y DZD revenue threatened

## 4. Visualisation RFM

In [13]:
fig = px.scatter(
    rfm,
    x='recence',
    y='frequence',
    size='montant',
    color='label',
    hover_data=['id_client', 'montant'],
    title='Segmentation Clients RFM — Energical',
    labels={
        'recence'  : 'Récence (jours)',
        'frequence': 'Fréquence (commandes)',
        'montant'  : 'Montant (DA)',
        'label'    : 'Segment'
    },
    color_discrete_map={
        'Champions 🟢': 'green',
        'A risque 🟡' : 'orange',
        'Perdus 🔴'   : 'red'
    }
)
fig.show()


## 5. CROSS-SELLING

In [14]:

from itertools import combinations
from collections import Counter

basket = df.groupby('id_commande_anon')['produit'].apply(list)

pairs = Counter()
for products in basket:
    for pair in combinations(set(products), 2):
        pairs[tuple(sorted(pair))] += 1

pairs_df = pd.DataFrame(pairs.items(), columns=['paire', 'frequence'])
pairs_df = pairs_df.sort_values('frequence', ascending=False).head(10)

print("\n" + "=" * 50)
print("TOP 10 CROSS-SELLING")
print("=" * 50)
print(pairs_df)



TOP 10 CROSS-SELLING
                                                  paire  frequence
2326  (Disjoncteur ECB5 DPN.1P+N.10, Disjoncteur ECB...         65
5693                  (Pressostat, Soupape de décharge)         60
3050      (Moteur vanne à 3 voies, Soupape de décharge)         55
2473  (Disjoncteur ECB5 DPN.1P+N.16, Disjoncteur ECB...         52
2133        (Vase d’expansion 10L, Vase d’expansion 8L)         51
5778               (Moteur vanne à 3 voies, Pressostat)         47
2456  (Disjoncteur ECB5 DPN.1P+N.10, Disjoncteur ECB...         41
2023  (Moniteur 4 fils noir 7 pouces VFE04, Sonnette...         40
5777  (Manomètre de pression d’eau 4 bar, Moteur van...         40
2026  (Serrure électrique noir avec doigt de verroui...         39


## 5. Export for the dashboard

In [15]:
pairs_df.to_csv('cross_selling.csv', index=False)
print("\n📁 Fichier exporté: cross_selling.csv")

rfm.to_csv('rfm_clients.csv', index=False)
print("\n📁 rfm_clients.csv")


📁 Fichier exporté: cross_selling.csv

📁 rfm_clients.csv
